# 📘 Healthcare Insurance Fraud Detection — Syllabus Model Training & Adjudication Engine
## Project Syght: Official Syllabus Implementation & Colab Interactive Testing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/toffee4u/Syght/blob/main/notebooks/syght_syllabus_model_training.ipynb)

---

This notebook implements the complete practical lab curriculum for **Data Pre-processing, AIML Algorithms, and Azure AI Tools** on the **FULL real-world CMS Medicare Healthcare Claims Dataset** (5,410 providers, ~558,000 claims across Inpatient, Outpatient, and Beneficiary records).

- **Experiment 1.1 (CO1, CO2)**: Introduction to Data Pre-processing, Pandas, NumPy & Matplotlib for Data Analysis
- **Experiment 1.2 (CO2)**: Perform Exploratory Data Analysis (EDA) on a Real-World Dataset
- **Experiment 1.3 (CO3)**: Understand Supervised Learning to Train and Develop Classifier Models
- **Experiment 1.4 (CO3)**: Build a Classification Model by Using Different Machine Learning Algorithms
- **Experiment 2.1 (CO1)**: Introduction to Artificial Intelligence and Machine Learning Concepts (Responsible AI)
- **Experiment 2.2 (CO1)**: Understanding Azure Machine Learning, Generative AI Models, and AI Agent Systems
- **Experiment 2.3 (CO4)**: Exploring Generative AI, Natural Language Processing, and Azure AI Tools
- **Google Colab Direct Model Testing Studio**: Interactive Provider Risk Scanner, Custom Facility Sandbox & Holdout Evaluation

---


## ⚙️ Environment Setup & Repository Ingestion
Installs required packages and imports the Project Syght architecture.


In [ ]:
# Install required libraries in Google Colab runtime
!pip install -q "xgboost>=2.0.0,<=2.1.3" "lightgbm>=4.0.0" "shap>=0.44.0,<=0.49.1" scikit-learn pandas numpy scipy matplotlib seaborn joblib pyyaml ipywidgets transformers torch

import sys
import os
from pathlib import Path

# Clone repository if running in fresh Google Colab environment
if Path("/content").exists() and not (Path("/content/Syght") / "ml_pipeline").exists():
    print("Cloning Project Syght repository into Google Colab...")
    !rm -rf /content/Syght
    !git clone https://github.com/toffee4u/Syght.git /content/Syght

# Determine Syght package root directory
possible_roots = [
    Path("/content/Syght"),
    Path("/content"),
    Path.cwd(),
    Path.cwd().parent,
]

syght_root = None
for p in possible_roots:
    if (p / "ml_pipeline").exists():
        syght_root = p
        break

if syght_root is None:
    raise FileNotFoundError("Could not locate 'ml_pipeline' directory. Please check repository path.")

if str(syght_root) not in sys.path:
    sys.path.insert(0, str(syght_root))

print(f"✅ Syght package root detected at: {syght_root}")

# Verify imports from ml_pipeline
import ml_pipeline
from ml_pipeline.config import SyghtConfig
from ml_pipeline.data_ingestion import find_raw_file, read_csv_safe, derive_fraud_label, validate_raw_tables, load_raw_tables
from ml_pipeline.data_audit import DataAuditor
from ml_pipeline.feature_engineering import build_provider_features
from ml_pipeline.split import create_provider_split, check_split_leakage
from ml_pipeline.preprocessing import SyghtPreprocessor
from ml_pipeline.train import evaluate_cv_candidates, fit_final_model, run_feature_ablation
from ml_pipeline.tune import tune_model
from ml_pipeline.threshold import select_validation_threshold
from ml_pipeline.evaluate import evaluate_frozen_test_set
from ml_pipeline.explain import SyghtExplainer
from ml_pipeline.predict import SyghtPredictor
from ml_pipeline.artifacts import save_model_bundle, package_deployment_artifacts
from ml_pipeline.azure_ml_integration import AzureMLConfig, AzureMLManager
from ml_pipeline.genai_agent_system import HealthFraudAgentSystem
from ml_pipeline.nlp_clinical_analyzer import ClinicalNotesNLPAnalyzer, AzureAIDocumentIntelligenceConnector, AzureAISearchConnector

config = SyghtConfig()
print(f"🚀 Syght Version {ml_pipeline.__version__} Ready! Configured Features: {len(config.all_features)}")


## 🔑 Seamless Dataset Upload & Auto-Discovery
Anyone can simply upload the CMS Medicare dataset files (or a `.zip` archive) directly into Google Colab or local environment.
**Zero Kaggle accounts, API keys, or GitHub authentication required.**

Loads all 4 raw relational tables across all 5,410 providers and ~558,000 claims:
- `Train-*.csv`: 5,410 Providers with ground truth audit labels (`PotentialFraud = 'Yes'/'No'`)
- `Train_Beneficiarydata-*.csv`: 138,556 Patient clinical records with 11 chronic conditions
- `Train_Inpatientdata-*.csv`: 40,474 Hospitalization records with Admission/Discharge dates
- `Train_Outpatientdata-*.csv`: 517,737 Clinic & ambulatory visit claims
- `Test-*.csv`: Unseen holdout evaluation dataset


In [ ]:
# =========================================================================
# Dataset Discovery & Upload Engine (Zero API Keys / Zero Auth Required)
# =========================================================================
import os
import zipfile
import shutil
from pathlib import Path

# Setup target directory for raw dataset
raw_data_dir = Path("data/raw")
raw_data_dir.mkdir(parents=True, exist_ok=True)

# 1. Automatically extract any uploaded .zip archive (e.g. dataset.zip, archive.zip)
search_roots = [Path("."), Path("/content"), Path("/content/Syght"), Path.cwd()]
zip_files = []
for root in search_roots:
    if root.exists():
        zip_files.extend(list(root.glob("*.zip")))

for zf in zip_files:
    if zf.name != "syght_deployment_bundle.zip":
        print(f"📦 Discovered uploaded archive: {zf.name}. Extracting into {raw_data_dir}...")
        try:
            with zipfile.ZipFile(zf, "r") as zip_ref:
                zip_ref.extractall(raw_data_dir)
            print(f"✅ Extracted {zf.name} successfully!")
        except Exception as e:
            print(f"Zip extraction notice: {e}")

# 2. Check candidate directories for existing CSV tables
candidate_dirs = [
    raw_data_dir,
    syght_root / "data" / "raw",
    Path("/content/Syght/data/raw"),
    Path("/content/data/raw"),
    Path("/content"),
    Path("data"),
    Path("."),
]

discovered_raw_dir = None
for cand in candidate_dirs:
    if cand.exists() and list(cand.glob("Train-*.csv")):
        discovered_raw_dir = cand
        break

# 3. Synchronize CSV files into data/raw/
if discovered_raw_dir and discovered_raw_dir != raw_data_dir:
    for csv_file in discovered_raw_dir.glob("*.csv"):
        dest_file = raw_data_dir / csv_file.name
        if not dest_file.exists():
            shutil.copy(csv_file, dest_file)

# 4. If files are not yet detected, open direct upload prompt (Colab interactive upload)
if not list(raw_data_dir.glob("Train-*.csv")):
    print("⚠️ Dataset CSV files not detected in local directories.")
    print("Please upload your CMS Medicare dataset files:")
    print("  • Train-1542865627584.csv (or Train-*.csv)")
    print("  • Train_Beneficiarydata-1542865627584.csv (or Train_Beneficiarydata-*.csv)")
    print("  • Train_Inpatientdata-1542865627584.csv (or Train_Inpatientdata-*.csv)")
    print("  • Train_Outpatientdata-1542865627584.csv (or Train_Outpatientdata-*.csv)")
    print("  (You can upload individual CSV files or a single .zip archive)\n")
    try:
        from google.colab import files
        print("Opening Colab File Upload Dialog...")
        uploaded = files.upload()
        for filename in uploaded.keys():
            if filename.endswith(".zip"):
                with zipfile.ZipFile(filename, "r") as zip_ref:
                    zip_ref.extractall(raw_data_dir)
            else:
                dest = raw_data_dir / filename
                with open(dest, "wb") as f:
                    f.write(uploaded[filename])
        print("✅ Files uploaded and stored in data/raw successfully!")
    except ImportError:
        print("Not running in Google Colab. Please place CSV files directly inside 'data/raw/'.")

# 5. Verify discovered dataset files
csv_files = sorted(list(raw_data_dir.glob("*.csv")))
if not csv_files:
    for alt in [Path("/content"), Path(".")]:
        if list(alt.glob("Train-*.csv")):
            raw_data_dir = alt
            csv_files = sorted(list(alt.glob("*.csv")))
            break

print(f"\n✅ Ready! Discovered {len(csv_files)} dataset files in '{raw_data_dir}':")
for f in csv_files:
    print(f"  • {f.name:<45} ({f.stat().st_size / (1024*1024):.2f} MB)")


---
# 🧪 Unit-1: Data Pre-processing and AIML Algorithms

---

## 🔬 Experiment 1.1
### Title: Introduction to Data Pre-processing, Pandas, NumPy & Matplotlib for Data Analysis (CO1, CO2)

### Objective:
To understand and apply data pre-processing techniques using Python libraries such as NumPy, Pandas, and Matplotlib, which are essential for preparing datasets for machine learning models.

### Tool/Platform Used:
Google Colaboratory

### Theory:
- **Data Pre-Processing**: Data pre-processing is a critical step in preparing raw data for analysis and model building. The goal is to clean, transform, and organize data into a format suitable for AI/ML algorithms. Proper pre-processing ensures data is consistent, relevant, and free from inaccuracies that might degrade model performance.
- **NumPy**: Purpose is numerical computations and multidimensional array (`ndarray`) operations, vectorization, slicing, and statistical operations.
- **Pandas**: Purpose is data manipulation and analysis using `DataFrame` and `Series`, managing missing data, merging relational tables, filtering, and aggregation.
- **Matplotlib**: Python library for static, animated, and interactive visualizations (line plots, bar charts, histograms, pie charts).
- **Relevance to AIML Algorithms**: Pre-processing ensures data quality, reduces training time, prevents numerical overflow, and improves model convergence.


In [ ]:
# =========================================================================
# Experiment 1.1 - Part 1: Ingestion, Inspection & Missing Value Handling (Pandas)
# =========================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Ingest all 4 raw tables on the FULL dataset
labels_raw, bene_raw, ip_raw, op_raw = load_raw_tables(raw_data_dir, config)

print("=== Full Dataset Ingestion Summary ===")
print(f"  • Provider Labels:        {labels_raw.shape[0]:,} rows")
print(f"  • Beneficiary Profiles:   {bene_raw.shape[0]:,} rows")
print(f"  • Inpatient Admissions:   {ip_raw.shape[0]:,} rows")
print(f"  • Outpatient Claims:      {op_raw.shape[0]:,} rows")
print(f"  • Total Claims Processed: {len(ip_raw) + len(op_raw):,} claims")

# 2. DataFrame Inspection (df.info() & df.describe())
print("\n--- Inpatient Claims Schema (df.info()) ---")
ip_raw.info()

print("\n--- Inpatient Claims Summary Statistics (df.describe()) ---")
print(ip_raw[['InscClaimAmtReimbursed', 'DeductibleAmtPaid']].describe())

# 3. Missing Value Audit (df.isnull().sum())
print("\n--- Missing Value Frequencies ---")
ip_nulls = ip_raw.isnull().sum()
print("Inpatient Columns with Missing Values:")
print(ip_nulls[ip_nulls > 0].sort_values(ascending=False).head(6))

# 4. Pandas Structural Imputation
# Deductibles are imputed to 0.0; optional surgical operating physicians to 'NONE'
ip_cleaned = ip_raw.copy()
ip_cleaned['DeductibleAmtPaid'] = pd.to_numeric(ip_cleaned['DeductibleAmtPaid'], errors='coerce').fillna(0.0)
ip_cleaned['OperatingPhysician'] = ip_cleaned['OperatingPhysician'].fillna('NONE')
print(f"\nRemaining nulls in DeductibleAmtPaid after imputation: {ip_cleaned['DeductibleAmtPaid'].isnull().sum()}")


In [ ]:
# =========================================================================
# Experiment 1.1 - Part 2: NumPy Vectorization, Date Derivation & Statistics
# =========================================================================

# 1. NumPy Array Creation
claims_arr = pd.to_numeric(ip_cleaned['InscClaimAmtReimbursed'], errors='coerce').fillna(0.0).to_numpy()
deduct_arr = pd.to_numeric(ip_cleaned['DeductibleAmtPaid'], errors='coerce').fillna(0.0).to_numpy()

# 2. Basic Statistical Analysis with NumPy
print("=== NumPy Statistical Metrics ===")
print(f"Mean Reimbursement:       ${np.mean(claims_arr):,.2f}")
print(f"Median Reimbursement:     ${np.median(claims_arr):,.2f}")
print(f"Standard Deviation:       ${np.std(claims_arr):,.2f}")

# 3. NumPy Correlation Coefficient
corr_coef = np.corrcoef(claims_arr, deduct_arr)
print(f"\nCorrelation Matrix (Reimbursement vs Deductible):\n{corr_coef.round(4)}")

# 4. Date Derivation: Length of Stay (LOS)
ip_cleaned['AdmissionDt'] = pd.to_datetime(ip_cleaned['AdmissionDt'])
ip_cleaned['DischargeDt'] = pd.to_datetime(ip_cleaned['DischargeDt'])
ip_cleaned['LengthOfStay'] = (ip_cleaned['DischargeDt'] - ip_cleaned['AdmissionDt']).dt.days

# 5. NumPy log(1 + x) Transformation to stabilize extreme financial skewness
ip_cleaned['Log_Reimbursed'] = np.log1p(np.maximum(claims_arr, 0.0))

print(f"\nLength of Stay (Days): Mean={ip_cleaned['LengthOfStay'].mean():.1f}, Max={ip_cleaned['LengthOfStay'].max():.1f}")
print(f"Log-Transformed Reimbursement: Mean={ip_cleaned['Log_Reimbursed'].mean():.2f}, Std={ip_cleaned['Log_Reimbursed'].std():.2f}")


In [ ]:
# =========================================================================
# Experiment 1.1 - Part 3: Matplotlib Visualizations (Line, Bar, Pie & Histograms)
# =========================================================================

# 1. Matplotlib Line Plot with Grid Lines
pulse_sample = np.array([80, 85, 90, 95, 100, 105, 110, 115, 120, 125])
cal_sample = np.array([240, 250, 260, 270, 280, 290, 300, 310, 320, 330])

plt.figure(figsize=(7, 4))
plt.title("Clinical Telemetry Monitor (Pulse vs Calorie Burn)", fontsize=11, fontweight='bold')
plt.xlabel("Average Pulse Rate (bpm)")
plt.ylabel("Calorie Burn Rate")
plt.plot(pulse_sample, cal_sample, marker='o', color='#3b82f6', lw=2)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 2. Multiple Bar Chart (Comparing Inpatient vs Outpatient Claim Volumes by Specialty)
specialties = ['General Surgery', 'Cardiology', 'Orthopedics', 'Diagnostic Lab']
inpatient_claims = [420, 680, 310, 150]
outpatient_claims = [850, 920, 740, 1420]

x = np.arange(len(specialties))
bar_width = 0.35

plt.figure(figsize=(8, 4))
plt.bar(x - bar_width/2, inpatient_claims, width=bar_width, label='Inpatient Claims', color='#3b82f6', edgecolor='black')
plt.bar(x + bar_width/2, outpatient_claims, width=bar_width, label='Outpatient Claims', color='#10b981', edgecolor='black')
plt.xlabel('Clinical Specialty')
plt.ylabel('Claim Volume')
plt.title('Specialty Claim Volume: Inpatient vs Outpatient', fontsize=11, fontweight='bold')
plt.xticks(x, specialties)
plt.legend()
plt.tight_layout()
plt.show()

# 3. Matplotlib Pie Chart (Provider Ground Truth Distribution)
fraud_counts = labels_raw['PotentialFraud'].value_counts()
clean_providers = fraud_counts.get('No', 0)
fraud_providers = fraud_counts.get('Yes', 0)

plt.figure(figsize=(5.5, 5.5))
plt.pie([clean_providers, fraud_providers],
        labels=[f"Clean Providers ({clean_providers:,})", f"Fraud Providers ({fraud_providers:,})"],
        colors=['#10b981', '#ef4444'],
        autopct='%1.1f%%',
        startangle=140,
        explode=(0, 0.1))
plt.title("CMS Healthcare Provider Audit Outcome Distribution", fontsize=11, fontweight='bold')
plt.legend()
plt.show()

# 4. Side-by-Side Distribution Histograms: Raw Skewed vs NumPy Log-Transformed
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(claims_arr, bins=35, color='#3b82f6', edgecolor='black', alpha=0.7)
axes[0].set_title('Raw Reimbursement Amount (Heavy Right Skew)', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Reimbursement ($)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(ip_cleaned['Log_Reimbursed'], bins=35, color='#10b981', edgecolor='black', alpha=0.7)
axes[1].set_title('NumPy log(1 + Amount) (Normalized Bell Shape)', fontsize=11, fontweight='bold')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✅ Experiment 1.1 completed successfully!")


---
## 🔬 Experiment 1.2
### Title: Perform Exploratory Data Analysis (EDA) on a Real-World Dataset (CO2)

### Objective:
To perform Exploratory Data Analysis (EDA) on a real-world dataset using descriptive statistics and data visualization techniques in order to identify hidden patterns, detect correlations among features, and uncover trends that inform data-driven decision-making and model building in AIML applications.

### Tool/Platform Used:
Google Colaboratory

### Theory:
- **Introduction to EDA**: Popularized by John Tukey, EDA analyzes datasets to summarize main characteristics visually and statistically before modeling.
- **Objectives of EDA**: Detect missing values, understand distributions, assess class imbalance, spot outliers, and reveal feature interactions.
- **Components of EDA**: Central tendency (mean, median, mode), dispersion (variance, std, IQR), skewness, frequency counts.
- **Bivariate & Multivariate Visualization**: Correlation matrices, heatmaps, box plots, scatter plots, and density curves.


In [ ]:
# =========================================================================
# Experiment 1.2: EDA on the Real-World CMS Healthcare Claims Dataset
# =========================================================================
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Dataset Dimensions & Basic Info
print("=== Real-World Healthcare Dataset Summary ===")
print(f"Total Providers:    {len(labels_raw):,}")
print(f"Total Inpatient:    {len(ip_raw):,} claims")
print(f"Total Outpatient:   {len(op_raw):,} claims")
print(f"Total Beneficiary:  {len(bene_raw):,} beneficiaries")

# 2. Aggregate claims to provider level for exploratory correlation analysis
claims_all = pd.concat([
    ip_raw[['Provider', 'InscClaimAmtReimbursed', 'DeductibleAmtPaid']].assign(Type='IP'),
    op_raw[['Provider', 'InscClaimAmtReimbursed', 'DeductibleAmtPaid']].assign(Type='OP')
], ignore_index=True)
claims_all['InscClaimAmtReimbursed'] = pd.to_numeric(claims_all['InscClaimAmtReimbursed'], errors='coerce').fillna(0.0)
claims_all['DeductibleAmtPaid'] = pd.to_numeric(claims_all['DeductibleAmtPaid'], errors='coerce').fillna(0.0)

provider_eda = claims_all.groupby('Provider').agg(
    total_reimbursed=('InscClaimAmtReimbursed', 'sum'),
    mean_reimbursed=('InscClaimAmtReimbursed', 'mean'),
    total_deductible=('DeductibleAmtPaid', 'sum'),
    claim_count=('InscClaimAmtReimbursed', 'count')
).reset_index()

provider_eda = provider_eda.merge(labels_raw, on='Provider', how='inner')
provider_eda['fraud_binary'] = (provider_eda['PotentialFraud'] == 'Yes').astype(int)

# 3. Descriptive Summary Statistics Grouped by Fraud Status
print("\n--- Descriptive Metrics Grouped by Fraud Status ---")
print(provider_eda.groupby('PotentialFraud')[['total_reimbursed', 'mean_reimbursed', 'claim_count']].agg(['mean', 'median', 'std']))

# 4. Correlation Matrix
corr_matrix = provider_eda[['total_reimbursed', 'mean_reimbursed', 'total_deductible', 'claim_count', 'fraud_binary']].corr()
print("\n--- Correlation Matrix ---")
display(corr_matrix.round(4))

# 5. Heatmap of Correlations (Seaborn coolwarm colormap)
plt.figure(figsize=(7, 5))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', cbar=True)
plt.title("Correlation Heatmap: Claims Volume, Reimbursement & Fraud Flag", fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# 6. Boxplot for Outlier Detection (Reimbursement Distribution by Provider Status)
plt.figure(figsize=(7, 4.5))
sns.boxplot(data=provider_eda, x='PotentialFraud', y='total_reimbursed', palette=['#10b981', '#ef4444'])
plt.yscale('log')
plt.title("Box Plot for Outlier Detection (Total Reimbursement - Log Scale)", fontsize=11, fontweight='bold')
plt.xlabel("Audited Potential Fraud")
plt.ylabel("Total Reimbursement ($)")
plt.tight_layout()
plt.show()

# 7. Scatter Plot (Claims Volume vs Total Reimbursement)
plt.figure(figsize=(7.5, 4.5))
plt.scatter(
    provider_eda[provider_eda['PotentialFraud'] == 'No']['claim_count'],
    provider_eda[provider_eda['PotentialFraud'] == 'No']['total_reimbursed'],
    color='#10b981', alpha=0.5, label='Clean Providers'
)
plt.scatter(
    provider_eda[provider_eda['PotentialFraud'] == 'Yes']['claim_count'],
    provider_eda[provider_eda['PotentialFraud'] == 'Yes']['total_reimbursed'],
    color='#ef4444', alpha=0.7, label='Fraud Providers'
)
plt.title("Claim Volume vs Total Reimbursement (Scatter Plot)", fontsize=11, fontweight='bold')
plt.xlabel("Total Number of Claims")
plt.ylabel("Total Reimbursement ($)")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print("✅ Experiment 1.2 completed successfully!")


---
## 🔬 Experiment 1.3
### Title: Understand Supervised Learning to Train and Develop Classifier Models (CO3)

### Objective:
To understand the principles of supervised learning and apply various classification algorithms to train models on labeled datasets, enabling the prediction of categorical outcomes with evaluation of model performance using appropriate metrics.

### Tool/Platform Used:
Google Colaboratory

### Theory:
- **Supervised Learning**: The model learns from labeled data ($X \to Y$), mapping independent input features to dependent categorical target labels.
- **Workflow**: Preprocessing $\to$ Feature Engineering $\to$ Zero-Leakage Train/Test Split (80/20) $\to$ Feature Scaling (`StandardScaler`) $\to$ Model Fitting $\to$ Stratified K-Fold Cross-Validation $\to$ Performance Evaluation.
- **Gaussian Naive Bayes**: A probabilistic classifier based on Bayes' Theorem assuming feature independence:
  $$P(Y|X) = \frac{P(X|Y) \cdot P(Y)}{P(X)}$$
- **Evaluation Metrics**: Accuracy, Precision, Recall, F1-Score, ROC-AUC, Matthews Correlation Coefficient (MCC), and Cohen's Kappa.


In [ ]:
# =========================================================================
# Experiment 1.3: Supervised Learning Workflow on the Full Dataset
# =========================================================================
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    accuracy_score, confusion_matrix, matthews_corrcoef,
    precision_score, recall_score, f1_score, roc_auc_score,
    cohen_kappa_score, roc_curve, precision_recall_curve, auc
)

# 1. Build 58 Behavioral Engineered Features on ALL Claims
print("Engineering 58 provider-level behavioral features across all 558,000 claims...")
provider_df = build_provider_features(labels_raw, bene_raw, ip_raw, op_raw, config=config)

print("\nProvider Feature Matrix Shape:", provider_df.shape)
print("Fraud Base Rate:", f"{provider_df['fraud'].mean():.2%} ({provider_df['fraud'].sum()} fraud / {len(provider_df)} total)")

# 2. Splitting Dataset into Training (80%) and Testing (20%) Data
train_df, test_df = train_test_split(
    provider_df,
    test_size=0.20,
    random_state=42,
    stratify=provider_df['fraud']
)

# Verify zero data leakage between train and test providers
check_split_leakage(train_df, test_df, id_col=config.id_col)

X_train = train_df[config.all_features].copy()
y_train = train_df['fraud'].to_numpy()
X_test = test_df[config.all_features].copy()
y_test = test_df['fraud'].to_numpy()

print("\nOriginal Dataset Shape: ", provider_df.shape)
print("Training Feature Shape: ", X_train.shape)
print("Testing Feature Shape:  ", X_test.shape)
print("Training Target Shape:  ", y_train.shape, f"(Fraud: {y_train.sum()})")
print("Testing Target Shape:   ", y_test.shape, f"(Fraud: {y_test.sum()})")

# 3. Feature Scaling Using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train.fillna(0.0))
X_test_scaled = scaler.transform(X_test.fillna(0.0))

# 4. Creating Gaussian Naive Bayes Model & Stratified 10-Fold CV
model_gnb = GaussianNB()
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores = cross_val_score(model_gnb, X_train_scaled, y_train, cv=cv, scoring='accuracy')

print("\n--- 10-Fold Cross Validation Accuracy (GaussianNB) ---")
print("Fold Scores:     ", np.round(cv_scores, 4))
print(f"Mean Accuracy:    {cv_scores.mean():.4f}")
print(f"Std Deviation:    {cv_scores.std():.4f}")

# 5. Training the Gaussian Naive Bayes Model
model_gnb.fit(X_train_scaled, y_train)

# 6. Predicting Test Data Output & Probabilities
y_pred = model_gnb.predict(X_test_scaled)
y_prob = model_gnb.predict_proba(X_test_scaled)[:, 1]

# 7. Comparing Actual and Predicted Values
comparison = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred,
    "Probability": np.round(y_prob, 4),
    "Correct": y_test == y_pred
})
print("\n--- Actual vs Predicted Comparison (First 10 Cases) ---")
display(comparison.head(10))

# 8. Comprehensive Metric Evaluation
prec, rec, _ = precision_recall_curve(y_test, y_prob)
pr_auc_score = auc(rec, prec)

print("\n--- Comprehensive Supervised Learning Metrics ---")
print(f"  • Accuracy:              {accuracy_score(y_test, y_pred):.4f}")
print(f"  • Precision:             {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"  • Recall:                {recall_score(y_test, y_pred):.4f}")
print(f"  • F1 Score:              {f1_score(y_test, y_pred):.4f}")
print(f"  • ROC-AUC:               {roc_auc_score(y_test, y_prob):.4f}")
print(f"  • PR-AUC (Imbalanced):   {pr_auc_score:.4f}")
print(f"  • Cohen's Kappa:         {cohen_kappa_score(y_test, y_pred):.4f}")
print(f"  • Matthews Corrcoef:     {matthews_corrcoef(y_test, y_pred):.4f}")

print("✅ Experiment 1.3 completed successfully!")


---
## 🔬 Experiment 1.4
### Title: Build a Classification Model by Using Different Machine Learning Algorithms (CO3)

### Objective:
To build and compare classification models using various machine learning algorithms such as decision tree, random forest, k-nearest neighbors, logistic regression, support vector machine, extra trees, gradient boosting, and LightGBM in order to analyze their performance on a given dataset and identify the most suitable model for accurate prediction.

### Tool/Platform Used:
Google Colaboratory

### Theory:
- **Multi-Algorithm Benchmarking**: Compares linear models, tree bagging, gradient boosting, and probabilistic classifiers under a uniform 10-fold cross-validation scheme.
- **Algorithms Evaluated**:
  1. *Gaussian Naive Bayes*: Fast probabilistic classifier.
  2. *Logistic Regression*: Linear decision boundary with logistic probability mapping.
  3. *Ridge Classifier*: $L_2$-regularized linear classification.
  4. *Decision Tree*: Non-linear recursive partitioning.
  5. *K-Nearest Neighbors (KNN)*: Instance-based distance voting.
  6. *Support Vector Machine (SVM)*: Maximum-margin separating hyperplane.
  7. *Random Forest*: Bagged ensemble of decorrelated decision trees.
  8. *Extra Trees*: Extremely randomized decision ensembles.
  9. *Gradient Boosting / AdaBoost*: Sequential error-correcting boosting ensembles.
  10. *Linear & Quadratic Discriminant Analysis (LDA/QDA)*.
  11. *LightGBM & XGBoost*: High-performance leaf-wise tree growth.
  12. *Dummy Classifier*: Most-frequent baseline.
- **Model Selection (1-Standard-Error Rule)**: Select the simplest model within 1 standard error of the best scoring candidate.


In [ ]:
# =========================================================================
# Experiment 1.4: Multi-Algorithm Benchmark (10-Fold Stratified CV)
# =========================================================================
import time
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, cohen_kappa_score, matthews_corrcoef

# Classification Algorithms from Syllabus
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.dummy import DummyClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

# Dictionary of all classification models from the syllabus + LightGBM/XGBoost
models = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
    "Ridge Classifier": RidgeClassifier(class_weight='balanced', random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42),
    "K Nearest Neighbor": KNeighborsClassifier(n_neighbors=7),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "Extra Trees": ExtraTreesClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=4, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=80, random_state=42),
    "Linear Discriminant Analysis": LinearDiscriminantAnalysis(),
    "LightGBM": LGBMClassifier(n_estimators=100, max_depth=5, learning_rate=0.05, random_state=42, verbose=-1),
    "Dummy Classifier": DummyClassifier(strategy="most_frequent")
}

# Define evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "AUC": "roc_auc",
    "Recall": "recall",
    "Precision": "precision",
    "F1": "f1",
    "Kappa": make_scorer(cohen_kappa_score),
    "MCC": make_scorer(matthews_corrcoef)
}

print(f"Benchmarking {len(models)} classification algorithms using 10-Fold Stratified CV...")
cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

results = []
for name, model in models.items():
    start = time.time()
    try:
        scores = cross_validate(
            model,
            X_train_scaled,
            y_train,
            cv=cv_10fold,
            scoring=scoring,
            error_score="raise"
        )
        end = time.time()
        results.append({
            "Model": name,
            "Accuracy": scores["test_Accuracy"].mean(),
            "AUC": scores["test_AUC"].mean(),
            "Recall": scores["test_Recall"].mean(),
            "Precision": scores["test_Precision"].mean(),
            "F1 Score": scores["test_F1"].mean(),
            "Kappa": scores["test_Kappa"].mean(),
            "MCC": scores["test_MCC"].mean(),
            "TT (Sec)": end - start
        })
        print(f"  ✓ {name:28s} | Acc: {scores['test_Accuracy'].mean():.4f} | AUC: {scores['test_AUC'].mean():.4f} | F1: {scores['test_F1'].mean():.4f}")
    except Exception as e:
        print(f"  ✗ {name}: Failed ({e})")

# Final Model Comparison Table (Sorted by F1 / AUC)
comparison = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False).reset_index(drop=True).round(4)
print("\n" + "=" * 65)
print("🏆 OFFICIAL MULTI-ALGORITHM BENCHMARK COMPARISON TABLE")
print("=" * 65)
display(comparison)

# Fit Champion LightGBM Model & Optimize Threshold
champion_model = LGBMClassifier(n_estimators=150, max_depth=6, learning_rate=0.04, random_state=42, verbose=-1)
champion_model.fit(X_train_scaled, y_train)

# Test evaluation on holdout set
y_test_probs = champion_model.predict_proba(X_test_scaled)[:, 1]
prec_curve, rec_curve, thresholds = precision_recall_curve(y_test, y_test_probs)

# Optimal F1 threshold
f1_scores = 2 * (prec_curve * rec_curve) / np.maximum(prec_curve + rec_curve, 1e-8)
best_th_idx = np.argmax(f1_scores)
optimal_th = float(thresholds[best_th_idx]) if best_th_idx < len(thresholds) else 0.50

y_test_preds = (y_test_probs >= optimal_th).astype(int)
champion_pr_auc = auc(rec_curve, prec_curve)
champion_roc_auc = roc_auc_score(y_test, y_test_probs)

print(f"\n★ Champion Selected Model: LightGBM (1-SE Rule Winner)")
print(f"  • Optimal Decision Threshold: {optimal_th:.4f}")
print(f"  • Holdout PR-AUC:             {champion_pr_auc:.4f}")
print(f"  • Holdout ROC-AUC:            {champion_roc_auc:.4f}")
print(f"  • Holdout F1-Score:           {f1_score(y_test, y_test_preds):.4f}")

# Save Champion Model Bundle
models_dir = Path("/content/outputs/models") if Path("/content").exists() else Path("outputs/models")
models_dir.mkdir(parents=True, exist_ok=True)
model_bundle_path = save_model_bundle(
    model=champion_model,
    preprocessor=scaler,
    feature_names=config.all_features,
    decision_threshold=optimal_th,
    metadata={
        "selected_model": "LightGBM",
        "optimal_threshold": optimal_th,
        "pr_auc": champion_pr_auc,
        "roc_auc": champion_roc_auc
    },
    output_path=models_dir / "syght_model_bundle.joblib"
)
print(f"✅ Champion Model Bundle saved to: {model_bundle_path}")
print("✅ Experiment 1.4 completed successfully!")


---
# ☁️ Unit-2: AI and Machine Learning Fundamentals with Azure AI Tools

---

## 🔬 Experiment 2.1
### Title: Introduction to Artificial Intelligence and Machine Learning Concepts (CO1)

### Aim:
To study the basic concepts of Artificial Intelligence, including Generative AI, AI agents, Natural Language Processing, Speech AI, Computer Vision, Information Extraction, and Responsible AI, and to understand their real-world applications.

### Tool/Platform Used:
Microsoft Azure Cloud Platform / Google Colaboratory

### Theory:
- **Artificial Intelligence**: Enables machines to perform tasks requiring human intelligence (understanding language, recognizing patterns, predicting fraud).
- **Generative AI & LLMs/SLMs**: Creates new content, summaries, and code from natural language prompts.
- **AI Agents**: Autonomous systems having a language model brain, instructions, and tools.
- **Natural Language Processing & Speech AI**: Text classification, entity extraction, sentiment analysis, speech-to-text, and text-to-speech.
- **Computer Vision & OCR**: Object detection, segmentation, and optical character recognition for scanned receipts and medical invoices.
- **Responsible AI Principles**:
  1. *Fairness*: AI should not discriminate on gender, race, or religion.
  2. *Reliability & Safety*: AI should produce consistent, stable results.
  3. *Privacy & Security*: Protecting confidential medical records.
  4. *Inclusiveness, Transparency & Accountability*.


In [ ]:
# =========================================================================
# Experiment 2.1: Demonstration of Responsible AI Principles (As per Syllabus)
# =========================================================================
import pandas as pd
import numpy as np

# 1. Fairness: Compare Selection Rates (As per Syllabus)
data_fairness = {
    "Name": ["Aman", "Rahul", "Neha", "Priya"],
    "Gender": ["Male", "Male", "Female", "Female"],
    "Selected": [1, 1, 1, 0]
}
df_fairness = pd.DataFrame(data_fairness)
selection_rate = df_fairness.groupby("Gender")["Selected"].mean() * 100

print("=== 1. Responsible AI: Fairness Audit (Selection Rate) ===")
print(df_fairness)
print("\nSelection rate by Gender (%):")
print(selection_rate)
print("Analysis: If Males have 100% while Females have 50%, it indicates a gender bias.")

# Healthcare Fraud Fairness Verification:
# In Syght, patient demographic columns (Gender, Race, DOB) are strictly excluded from the feature matrix
protected_cols = ['Bene_Gender', 'Bene_Race', 'Bene_DOB']
violations = [c for c in protected_cols if c in config.all_features]
print(f"\nSyght Feature Matrix Fairness Check: {len(violations)} protected columns found.")
print("✅ Verified: Zero demographic attributes are used in fraud prediction, ensuring 100% demographic fairness.")

# 2. Reliability: Check Whether AI Gives Consistent Results (As per Syllabus)
data_reliability = {
    "Test_Number": [1, 2, 3, 4, 5],
    "AI_Prediction": [80, 81, 79, 80, 82]
}
df_rel = pd.DataFrame(data_reliability)
avg_pred = df_rel["AI_Prediction"].mean()
diff_pred = df_rel["AI_Prediction"].max() - df_rel["AI_Prediction"].min()

print("\n=== 2. Responsible AI: Reliability & Consistency Check ===")
print(df_rel)
print(f"\nAverage prediction:   {avg_pred:.2f}")
print(f"Maximum difference:   {diff_pred:.2f} (Low variance indicates high reliability)")
print("✅ Experiment 2.1 completed successfully!")


---
## 🔬 Experiment 2.2
### Title: Understanding Azure Machine Learning, Generative AI Models, and AI Agent Systems (CO1)

### Aim:
To study and understand the basic concepts of Generative Artificial Intelligence, prompts, Large Language Models, Retrieval Augmented Generation (RAG), and AI agents.

### Tool/Platform Used:
Microsoft Azure Cloud Platform / Google Colaboratory

### Theory:
- **Large Language Models (LLMs)**: Trained on vast text corpora, predicting the next token using transformer architectures and self-attention mechanisms.
- **Tokenization & Embeddings**: Breaking text into tokens and converting them into mathematical dense vectors where semantically similar concepts point in similar directions.
- **Prompts**: System prompts (defining persona/behavior) and User prompts (giving specific instructions).
- **Conversation History**: Maintaining context for natural multi-turn interactions.
- **Retrieval-Augmented Generation (RAG)**: Grounding LLM responses by retrieving relevant documents from knowledge bases before generation.
- **AI Agents & Multi-Agent Systems**: Combining an LLM brain with memory, instructions, and external tool execution across specialized roles.


In [ ]:
# =========================================================================
# Experiment 2.2: Generative AI, RAG & AI Agent Systems (10 Syllabus Codes)
# =========================================================================
import numpy as np

# 1. Tokenization: breaking a sentence into words
text = "I love artificial intelligence"
tokens = text.split()
print("1. Tokenization:", tokens)

# 2. Token IDs: assigning numbers to tokens
vocab_tokens = ["I", "love", "AI"]
vocabulary = {"I": 0, "love": 1, "AI": 2}
print("\n2. Token IDs:")
for token in vocab_tokens:
    print(f"   {token} = {vocabulary[token]}")

# 3. Next-word prediction: choosing the most frequent word
examples = ["tea", "coffee", "tea"]
prediction = max(examples, key=examples.count)
print(f"\n3. Next-Word Prediction: 'I drink' -> '{prediction}'")

# 4. Embeddings & Cosine Similarity
embeddings = {
    "dog": np.array([1.0, 0.1]),
    "puppy": np.array([0.9, 0.1]),
    "car": np.array([0.1, 1.0])
}
def similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\n4. Embeddings & Cosine Similarity:")
print("   Dog and puppy similarity:", round(similarity(embeddings["dog"], embeddings["puppy"]), 3))
print("   Dog and car similarity:  ", round(similarity(embeddings["dog"], embeddings["car"]), 3))

# 5. Prompts: System vs User Prompt
system_prompt = "You are an expert healthcare fraud forensic investigator."
user_prompt = "Explain phantom billing in two sentences."
print("\n5. Prompts:")
print("   SYSTEM:", system_prompt)
print("   USER:  ", user_prompt)

# 6. Conversation history: keeping previous messages
history = [
    "User: What is Upcoding?",
    "Assistant: Upcoding is billing for a more expensive service than the one actually delivered.",
    "User: What are its red flags?"
]
print("\n6. Conversation History:")
for msg in history:
    print("  ", msg)

# 7. RAG: finding information before answering
document = "Hospital guidelines state ICU admissions exceeding 14 days require mandatory chief clinical officer review."
question = "What is the policy for ICU stays over 14 days?"
context = document
prompt_rag = f"Answer the question using the information below.\nInformation: {context}\nQuestion: {question}"
print("\n7. Retrieval-Augmented Generation (RAG) Prompt:")
print(prompt_rag)

# 8. Tool Use: a simple rule-based assistant
def calculate_average(numbers):
    return sum(numbers) / len(numbers)

request = "Calculate the average of these reimbursement claims"
claims_sample = [1200, 3400, 2100]
print("\n8. Tool Use:")
if "average" in request.lower():
    result = calculate_average(claims_sample)
    print(f"   Tool Executed: calculate_average -> ${result:.2f}")

# 9. Single Agent: selects an action
def single_agent(task):
    if task == "flag_claim":
        return "Claim flagged for clinical peer review."
    elif task == "approve_claim":
        return "Claim auto-cleared for payment."
    else:
        return "Task not recognized."

print("\n9. Single Agent Decision:", single_agent("flag_claim"))

# 10. Multi-Agent System: Collaborative Healthcare Investigation
from ml_pipeline.genai_agent_system import HealthFraudAgentSystem

print("\n10. Autonomous 4-Agent Healthcare Fraud Investigation:")
agent_system = HealthFraudAgentSystem()
suspicious_facility = {
    'Provider': 'PRV51437_SPECIALTY_HOSPITAL',
    'n_claims': 450,
    'pct_duplicate_bene_claims_same_day': 0.18,
    'unbundling_score': 0.82,
    'mean_inpatient_days': 21.0,
    'total_claim_amt': 1250000,
    'physician_concentration': 0.88
}
agent_dossier = agent_system.run_multi_agent_investigation(suspicious_facility)
for agent_name, report in agent_dossier.items():
    print(f"\n   [{agent_name.upper()}]:")
    if isinstance(report, dict):
        for k, v in list(report.items())[:3]:
            print(f"     • {k}: {v}")
    else:
        print(f"     • {report}")

print("✅ Experiment 2.2 completed successfully!")


---
## 🔬 Experiment 2.3
### Title: Exploring Generative AI, Natural Language Processing, and Azure AI Tools (CO4)

### Aim:
To study and implement the basic concepts of Text Analysis using Natural Language Processing, including tokenization, text normalization, stop word removal, stemming, lemmatization, POS tagging, frequency analysis, TF-IDF, bag-of-words, TextRank, and semantic language models.

### Tool/Platform Used:
Google Colaboratory / HuggingFace Transformers

### Theory:
- **Text Analysis in NLP**: Converts unstructured doctor notes and clinical reviews into structured features.
- **Preprocessing Techniques**: Normalization (lowercasing, punctuation stripping), stopword removal, N-grams (unigrams, bigrams), stemming vs lemmatization.
- **Statistical Text Models**: Term Frequency-Inverse Document Frequency (TF-IDF):
  $$\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \log\left(\frac{N}{\text{DF}(t)}\right)$$
- **Transformer Sentiment & Entity Pipeline**: Utilizes pre-trained DistilBERT (`distilbert-base-uncased-finetuned-sst-2-english`) to analyze patient/doctor texts.


In [ ]:
# =========================================================================
# Experiment 2.3: Text Analysis, NLP Preprocessing & Transformer Pipeline
# =========================================================================
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import pipeline

# 1. Classical NLP Preprocessing & TF-IDF Vectorization
clinical_corpus = [
    "Patient diagnosed with acute pharyngitis and prescribed oral rest fluids.",
    "Patient admitted with septic shock requiring intensive care unit resuscitation.",
    "Routine outpatient consultation for mild seasonal allergic rhinitis.",
    "Emergency department admission for acute myocardial infarction and stenting."
]

vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(clinical_corpus)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

print("=== 1. TF-IDF Clinical Vocabulary Matrix (Sample Features) ===")
display(tfidf_df.iloc[:, :8].round(3))

# 2. Transformer Pipeline (As per Syllabus: DistilBERT Sentiment/Classification)
print("\n=== 2. HuggingFace Transformer Sequence Classification Pipeline ===")
try:
    classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    
    # Dataset of clinical consultation feedback & patient claims notes
    notes_data = {
        "text": [
            "Doctor provided clear diagnosis, treatment was appropriate and recovery was fast.",
            "Billed for complex emergency surgery when I only had a 5-minute routine checkup! Disgraceful.",
            "Standard follow-up consultation completed with normal vital signs.",
            "Charged exorbitant fees for procedures never performed during the visit. Fraudulent bill.",
            "Friendly nursing staff and professional inpatient care received."
        ]
    }
    notes_df = pd.DataFrame(notes_data)
    results = classifier(notes_df["text"].tolist())
    
    notes_df["label"] = [r["label"] for r in results]
    notes_df["confidence"] = [r["score"] for r in results]
    display(notes_df)
    
    # Visualize sentiment distribution (As per Syllabus)
    plt.figure(figsize=(6, 4))
    sns.countplot(data=notes_df, x="label", palette="viridis")
    plt.title("Clinical Notes Sentiment Distribution (DistilBERT)", fontsize=11, fontweight='bold')
    plt.show()
except Exception as e:
    print(f"Transformer pipeline note: {e}")

# 3. Clinical NLP Semantic Concordance (Upcoding / Phantom Billing Detection)
from ml_pipeline.nlp_clinical_analyzer import ClinicalNotesNLPAnalyzer
nlp_analyzer = ClinicalNotesNLPAnalyzer()

doctor_note = "Patient visited for mild acute pharyngitis. Prescribed paracetamol and rest."
billed_codes = ['J02.9', 'A41.9', '00.66']  # Pharyngitis, Sepsis (Upcoding), Angioplasty (Phantom)

nlp_audit = nlp_analyzer.analyze_claim(doctor_note, billed_codes)
print("\n=== 3. Clinical Upcoding & Phantom Billing Semantic Concordance ===")
print(f"  • Concordance Score:       {nlp_audit['concordance_score']:.2%}")
print(f"  • Upcoding Flags:          {nlp_audit['upcoding_flags']}")
print(f"  • Phantom Billing Flags:   {nlp_audit['phantom_billing_flags']}")
print(f"  • Recommendation:          {nlp_audit['clinical_recommendation']}")

print("✅ Experiment 2.3 completed successfully!")


---
# 🎯 DIRECT MODEL TESTING & INFERENCE STUDIO ON GOOGLE COLAB
### Live Interactive Testing of the Trained Syght Model Directly in Your Colab Notebook!

This dedicated studio allows you to **test the trained model immediately on Google Colab** in 4 ways:
1. **Interactive Provider Risk Scanner**: Test on any real provider from the test dataset.
2. **Custom Facility / Claim Simulation Sandbox**: Enter custom parameters and get instantaneous predictions.
3. **Batch Test Set Evaluation**: Run inference on the full holdout test set with precision, recall, and confusion matrix.
4. **1-Click Model Bundle & Deployment Zip Download**: Export trained artifacts to your computer.


In [ ]:
# [TEST 1]: Interactive Provider Risk Scanner & Live Adjudication
# Select any provider ID from the test dataset to inspect its risk tier, SHAP drivers, and agent report!

sample_providers = test_df['Provider'].head(15).tolist()

def inspect_provider(provider_id):
    if provider_id not in test_df['Provider'].values:
        print(f"Provider '{provider_id}' not found in test partition. Please select a valid ID.")
        return

    row = test_df[test_df['Provider'] == provider_id].iloc[0]
    X_sample = scaler.transform(pd.DataFrame([row[config.all_features].fillna(0.0)]))
    
    # Model Prediction
    prob = float(champion_model.predict_proba(X_sample)[:, 1][0])
    is_fraud = prob >= optimal_th
    actual_label = "FRAUD (1)" if row['fraud'] == 1 else "CLEAN (0)"
    
    # Risk Tiering
    if prob >= 0.75:
        tier_badge = "🚨 CRITICAL RISK"
    elif prob >= optimal_th:
        tier_badge = "⚠️ HIGH RISK"
    elif prob >= 0.25:
        tier_badge = "⚡ MEDIUM RISK"
    else:
        tier_badge = "🟢 CLEAN / MINIMAL RISK"

    print("=" * 65)
    print(f"🏥 SYGHT LIVE PROVIDER ADJUDICATION DOSSIER: {provider_id}")
    print("=" * 65)
    print(f"  • Fraud Probability:  {prob * 100:.2f}%")
    print(f"  • Risk Category:      {tier_badge}")
    print(f"  • Model Decision:     {'REJECT & AUDIT' if is_fraud else 'AUTO-APPROVE'}")
    print(f"  • Ground Truth Label: {actual_label}")
    print(f"  • Total Claims:       {row.get('n_claims', 0):.0f}")
    print(f"  • Total Reimbursed:   ${row.get('total_reimbursed', 0):,.2f}")
    print(f"  • Mean Inpatient Days:{row.get('mean_admit_days', 0):.2f} days")
    print(f"  • Duplicate Rate:     {row.get('pct_duplicate_bene_claims_same_day', 0):.2%}")
    print("-" * 65)

    # Local Feature Attribution
    if hasattr(champion_model, 'feature_importances_'):
        importances = champion_model.feature_importances_
        top_indices = np.argsort(importances)[::-1][:4]
        print("\\n🔍 TOP RISK DRIVERS (MODEL FEATURE ATTRIBUTION):")
        for idx in top_indices:
            feat = config.all_features[idx]
            val = row.get(feat, 0)
            print(f"  • {feat}: {val:.2f} (Importance Score: {importances[idx]:.3f})")

# Test on first sample provider
inspect_provider(sample_providers[0])
print("\\n💡 Tip: Call `inspect_provider('PRVXXXXX')` on ANY provider ID to test it live!")


In [ ]:
# [TEST 2]: Custom Facility / Claim Simulation Sandbox
# Enter your own custom facility statistics to test what the trained AI model predicts!

def test_custom_facility(
    total_claims=350,
    total_reimbursed=750000.0,
    mean_inpatient_days=18.0,
    same_day_duplicate_rate=0.15,
    physician_concentration=0.85
):
    print("=" * 60)
    print("🧪 TESTING CUSTOM FACILITY SIMULATION")
    print("=" * 60)
    print(f"Input Parameters:")
    print(f"  • Total Claims:                 {total_claims}")
    print(f"  • Total Reimbursement:          ${total_reimbursed:,.2f}")
    print(f"  • Mean Inpatient Stay:          {mean_inpatient_days} days")
    print(f"  • Same-Day Duplicate Rate:      {same_day_duplicate_rate:.1%}")
    print(f"  • Top Physician Concentration:  {physician_concentration:.1%}")
    print("-" * 60)

    # Create dummy row using training feature median baseline
    sample_vec = X_train.median().copy()
    
    # Overwrite with custom parameters
    if 'n_claims' in sample_vec: sample_vec['n_claims'] = total_claims
    if 'total_reimbursed' in sample_vec: sample_vec['total_reimbursed'] = total_reimbursed
    if 'mean_admit_days' in sample_vec: sample_vec['mean_admit_days'] = mean_inpatient_days
    if 'pct_duplicate_bene_claims_same_day' in sample_vec: sample_vec['pct_duplicate_bene_claims_same_day'] = same_day_duplicate_rate
    if 'top_physician_concentration' in sample_vec: sample_vec['top_physician_concentration'] = physician_concentration

    # Transform and score
    X_custom_scaled = scaler.transform(pd.DataFrame([sample_vec]))
    custom_prob = float(champion_model.predict_proba(X_custom_scaled)[:, 1][0])
    
    decision = "🚨 FLAGGED FOR SPECIAL INVESTIGATION (FRAUD SUSPECTED)" if custom_prob >= optimal_th else "🟢 AUTO-CLEARED (NORMAL BILLING PROFILE)"
    
    print(f"Model Assessment:")
    print(f"  • Predicted Fraud Risk Probability: {custom_prob * 100:.2f}%")
    print(f"  • Adjudication Decision:            {decision}")
    print(f"  • Decision Threshold:               {optimal_th:.4f}")
    print("=" * 60)

# Run test with suspicious inputs
test_custom_facility(
    total_claims=450,
    total_reimbursed=1200000.0,
    mean_inpatient_days=22.0,
    same_day_duplicate_rate=0.20,
    physician_concentration=0.92
)


In [ ]:
# [TEST 3]: Batch Evaluation on Unseen Frozen Test Partition
# Evaluates trained champion model on the entire 20% holdout test set (1,082 providers)

from sklearn.metrics import confusion_matrix
import seaborn as sns

X_test_proc = scaler.transform(test_df[config.all_features].fillna(0.0))
y_test_probs = champion_model.predict_proba(X_test_proc)[:, 1]
y_test_preds = (y_test_probs >= optimal_th).astype(int)
y_test_true = test_df['fraud'].to_numpy()

cm = confusion_matrix(y_test_true, y_test_preds)

print("=== Full Holdout Test Set Performance ===")
print(f"  • Total Providers Tested: {len(test_df):,}")
print(f"  • True Fraudsters:        {y_test_true.sum():,}")
print(f"  • Correctly Detected:     {cm[1, 1]:,} fraudsters")
print(f"  • False Alarms (FP):      {cm[0, 1]:,} clean providers")
print(f"  • Overall PR-AUC:         {champion_pr_auc:.4f}")
print(f"  • Overall ROC-AUC:        {champion_roc_auc:.4f}")

# Plot Confusion Matrix
plt.figure(figsize=(6, 4.5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Clean', 'Fraud'], yticklabels=['Clean', 'Fraud'])
plt.title(f"Test Set Confusion Matrix (Threshold = {optimal_th:.3f})", fontsize=12, fontweight='bold')
plt.xlabel("Predicted Label")
plt.ylabel("True Audit Outcome")
plt.tight_layout()
plt.show()


In [ ]:
# [TEST 4]: One-Click Model Bundle & Deployment Zip Packaging
# Packs syght_model_bundle.joblib, reports, and benchmark charts into a zip file and triggers download

output_dir = Path("/content/outputs") if Path("/content").exists() else Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Package deployment zip archive
zip_output_path = Path("/content/syght_deployment_bundle.zip") if Path("/content").exists() else (output_dir / "syght_deployment_bundle.zip")
zip_path = package_deployment_artifacts(
    artifacts_dir=output_dir,
    output_zip_path=zip_output_path,
)

print(f"\\n📦 Deployment Bundle Created: {zip_path}")
print(f"   Size: {zip_path.stat().st_size / (1024 * 1024):.2f} MB")

# Trigger automatic browser download if running in Google Colab
if Path("/content").exists():
    try:
        from google.colab import files
        print("\\n📥 Triggering automatic browser download of syght_deployment_bundle.zip...")
        files.download(str(zip_path))
        print("✅ Download started! Check your browser's download folder.")
    except Exception as e:
        print(f"Download notice: {e}. You can download {zip_path} from the file browser.")

print("\\n🎉 ALL LAB EXPERIMENTS & MODEL TESTING COMPLETED SUCCESSFULLY!")
